# D1 · Reading a tree from the inside: nodes, paths, probabilities and rules

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Open a `DecisionTreeClassifier` trained on the Titanic and read its structure (`tree_`): number of nodes and leaves, root, purity of the leaves; rebuild `predict_proba` **by hand** by walking through the tree; find the leaf with the highest survival probability and write its rule in plain English.

## Dataset

`ds.titanic()` prepared in `X_tr, X_te, y_tr, y_te` with numeric columns: `pclass, age, fare` (missing values imputed with the median), `siblings_spouses`, `parents_children`, and the dummies `sex_male`, `title_Mr`, `title_Miss`, `title_Mrs`. Tree: `DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=0)`; `names` is the list of the column names.

## Educational purpose

Learn to read what scikit-learn hides: `tree_` is a handful of arrays (children, features, thresholds, values) and the whole model is a table. Reading it teaches you what a tree really is and how to explain a prediction ("this passenger ended up in that leaf because...").

## Task

1. Save `n_nodes`, `n_leaves`, `depth` (of the tree), `root` (name of the feature at the root), `root_threshold` and `root_samples` (number of examples at the root).
2. Write `proba_by_hand(tree, x)`: start from node 0 and go down following `children_left`/`children_right` (`x[feature] <= threshold` goes left) until a leaf (`children_left == -1`); return the class fractions of the leaf (`tree_.value` holds the **normalized** fractions in scikit-learn >= 1.4 and the counts in earlier versions: always normalize by dividing by the sum). Check that it matches `predict_proba` on the **whole** test set: save `matches` (True/False).
3. For each leaf compute `samples` (number of examples) and `purity` (fraction of the most frequent class). Save `largest_leaf` (node index of the leaf with the most samples) and its `largest_purity`, and `n_pure_leaves` (leaves with purity 1).
4. **Rule:** among the leaves with at least 20 samples find the one with the highest survival probability and save `rule` as a list of strings (one per condition, for example `"sex_male <= 0.5"`) obtained by climbing from the node to the root. Also save `rule_proba` (fraction of survivors in the leaf).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = ds.titanic()
y = df["survived"].to_numpy()
X = df[["pclass", "age", "fare", "siblings_spouses", "parents_children"]].copy()
X = X.fillna(X.median())                                                   # age and fare have a few missing values
X["sex_male"] = (df["sex"] == "male").astype(int)
for t in ["Mr", "Miss", "Mrs"]:
    X[f"title_{t}"] = (df["title"] == t).astype(int)
names = list(X.columns)
X_tr, X_te, y_tr, y_te = train_test_split(X.to_numpy(), y, test_size=0.3, stratify=y, random_state=0)
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=0).fit(X_tr, y_tr)
T = tree.tree_
print("columns:", names)

## Your work

In [ ]:
n_nodes = None
n_leaves = None
depth = None
root = None
root_threshold = None
root_samples = None

In [ ]:
def proba_by_hand(tree, x):
    ...

In [ ]:
matches = None

In [ ]:
largest_leaf = None
largest_purity = None
n_pure_leaves = None

In [ ]:
rule = None
rule_proba = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`T.children_left[n] == -1` identifies a leaf. Fraction: `v = T.value[n][0]; v / v.sum()`. Parent: build `parent = {}` with `for n in range(T.node_count): if T.children_left[n] != -1: parent[T.children_left[n]] = (n, "<="); parent[T.children_right[n]] = (n, ">")`. Rule: climb from the leaf to the root, for each step `f"{names[T.feature[p]]} {sign} {T.threshold[p]:.1f}"`, and reverse the order.'):
    check(n_nodes == T.node_count and n_leaves == tree.get_n_leaves() and depth == tree.get_depth() and n_nodes == 2 * n_leaves - 1, "nodes, leaves and depth read from tree_ (a binary tree has 2 * leaves - 1 nodes)", "T.node_count, get_n_leaves(), get_depth()")
    check(root == names[T.feature[0]] and close(root_threshold, T.threshold[0]) and root_samples == len(y_tr), "root, threshold and samples at the root (the whole training set)", "T.feature[0], T.threshold[0], T.n_node_samples[0]")
    check(matches is True, "proba_by_hand matches predict_proba on the whole test set", "go down with children_left/right; normalize tree_.value")
    leaves = np.flatnonzero(T.children_left == -1)
    samples = T.n_node_samples[leaves]
    check(largest_leaf == leaves[samples.argmax()] and samples.max() >= 100 and largest_purity > 0.7, "the largest leaf has over 100 examples (and purity above 0.7)", "n_node_samples of the leaves")
    purities = np.array([T.value[n][0].max() / T.value[n][0].sum() for n in leaves])
    check(n_pure_leaves == int((np.isclose(purities, 1.0)).sum()) and n_pure_leaves >= 1, "correct number of pure leaves (at least one)", "purity = max(value) / sum(value)")
    check(isinstance(rule, list) and len(rule) >= 3 and all(isinstance(r, str) and ("<=" in r or ">" in r) for r in rule) and rule_proba > 0.85, "rule with at least 3 conditions and survival probability over 0.85", "climb with a child -> parent dictionary")
    check(any("sex_male" in r or "title_Mr" in r for r in rule), "the rule involves sex (or title)", "the conditions of the safest leaf")
done()

---
The commented solution is in `solutions/D_interpret/D1_reading_the_structure_of_a_tree.ipynb`. Open it after you pass the check, or after you have really tried.